# Clements & Denolle (2023) — `codameter` pipeline demo (synthetic mode)

This notebook exercises the `codameter` pipeline against a **synthetic dataset** that mimics the on-disk layout of the [Clements & Denolle (2023)](https://doi.org/10.1029/2022JB025553) Zenodo archive.

> **No 4.4 GB download required.** The notebook generates a fake archive in `runs/cd2023_synthetic/synthetic_data/` and loads it via the same `load_clements_denolle_2023()` loader that real data would use. To switch to real data, set `DATA_DIR` to the path of the unpacked Zenodo archive.

### Data source (real mode)
```
Clements & Denolle (2023). The seismic signature of California's earthquakes, droughts, and floods.
JGR Solid Earth, 128, e2022JB025553. DOI: 10.1029/2022JB025553
Data: https://doi.org/10.5281/zenodo.6413275
```

In [ ]:
from __future__ import annotations
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pyarrow as pa
import pyarrow.feather as feather

%matplotlib inline
plt.rcParams.update({"figure.dpi": 130, "figure.figsize": (12, 4)})

from codameter import Site, run_workflow
from codameter.config import (
    AnalysisConfig, ForcingSpec, Forcings, Layer, Location,
    MaterialProperties, Measurement, Prior, VelocityModel,
)
from codameter.data.loaders import load_clements_denolle_2023, load_csv_timeseries
from codameter.forward.poroelastic import groundwater_level_okubo
from codameter.forward.thermoelastic import thermoelastic_dvv

print(f"codameter imported OK — Python {sys.version.split()[0]}")

## Configuration

Set `DATA_DIR = None` to run in **synthetic mode** (default).  
Point it at the unpacked Zenodo archive to use real data.

In [ ]:
STATION  = "CI.LJR"
DATA_DIR = None           # <- set to Path("/path/to/zenodo_archive") for real data
OUT_DIR  = Path("runs/cd2023_synthetic")
SEED     = 17

OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1 · Build the CI.LJR site configuration

In [ ]:
site = Site(
    site_id=f"cd2023_{STATION.replace('.', '_')}",
    location=Location(lat=34.55, lon=-117.77, elevation_m=830.0),
    measurement=Measurement(
        type="cross_correlation", frequency_band_hz=(0.5, 1.0),
    ),
    velocity_model=VelocityModel(
        layers=[
            Layer(thickness_km=0.20, vp=2.0, vs=0.8, rho=2.1),
            Layer(thickness_km=1.50, vp=4.0, vs=2.2, rho=2.4),
            Layer(thickness_km=5.00, vp=5.5, vs=3.1, rho=2.6),
            Layer(thickness_km=50.0, vp=6.2, vs=3.6, rho=2.8),
        ],
    ),
    forcings=Forcings(
        thermoelastic=ForcingSpec(
            enabled=True, model="phase_shift",
            extra={"time_shift_days": 50.0},
        ),
        hydrological=ForcingSpec(enabled=True, model="okubo_gwl"),
    ),
    material_properties=MaterialProperties(
        beta_prior=Prior(mean=300.0, std=100.0),
        mu_prime_prior=Prior(mean=280.0, std=90.0),
    ),
    analysis=AnalysisConfig(uncertainty_method="wls"),
)
print(site)

## 2 · Generate (or load) dv/v data and forcings

In [ ]:
YEAR_S = 365.25 * 86400.0
truth = None

real = (
    DATA_DIR is not None
    and DATA_DIR.exists()
    and any((DATA_DIR / "DVV").glob(f"{STATION}.*"))
)

if real:
    print(f"[mode] real data from {DATA_DIR}")
    dvv_data = load_clements_denolle_2023(DATA_DIR, STATION)
    forcings = {}   # add CSV paths here if available
else:
    target = OUT_DIR / "synthetic_data"
    target.mkdir(parents=True, exist_ok=True)
    print(f"[mode] synthetic — writing archive to {target}")

    rng = np.random.default_rng(SEED)
    n = 18 * 365
    times = pd.date_range("2002-01-01", periods=n, freq="D", tz="UTC")
    t_s = (times - times[0]).total_seconds().to_numpy()

    # Precipitation — winter storms, amplified 2004-05 event
    P = np.zeros(n)
    storm = (np.arange(n) % 365) > 300
    P[storm] = rng.lognormal(mean=-3.0, sigma=1.5, size=int(storm.sum()))
    winter_2005 = (times >= "2004-12-15") & (times <= "2005-02-25")
    P[winter_2005] *= 3.0

    # Temperature — annual cycle + noise
    T = 15.0 + 8.0 * np.sin(2 * np.pi * t_s / YEAR_S - 0.5) + 0.4 * rng.standard_normal(n)

    # Forward model
    p1_truth, p2_truth = -3.0e-3, 8.0e-5
    dGWL = groundwater_level_okubo(P, t_s, porosity=0.10,
                                   decay_rate_per_s=1.0 / (180 * 86400.0))
    T_pred = thermoelastic_dvv(T, t_s, sensitivity_amplitude=1.0, time_shift_days=50.0)
    dvv_true = p1_truth * (dGWL - dGWL.mean()) + p2_truth * T_pred
    dvv = dvv_true + 1.5e-4 * rng.standard_normal(n)
    truth = {"p1_dGWL": p1_truth, "p2_T": p2_truth}

    # Write feather (C&D layout: percent DVV)
    dvv_dir = target / "DVV"
    met_dir = target / "meteorology"
    dvv_dir.mkdir(exist_ok=True)
    met_dir.mkdir(exist_ok=True)

    df_feat = pd.DataFrame({
        "DATE": times.tz_convert(None),
        "DVV": dvv * 100.0,
        "DVV_ERR": np.full(n, 1.5e-4 * 100.0),
        "CC": rng.uniform(0.7, 0.95, n),
    })
    feather.write_feather(
        pa.Table.from_pandas(df_feat, preserve_index=False),
        dvv_dir / f"{STATION}.feather",
    )
    p_csv = met_dir / f"{STATION.replace('.', '_')}_P.csv"
    t_csv = met_dir / f"{STATION.replace('.', '_')}_T.csv"
    pd.DataFrame({"time": times, "precipitation": P}).to_csv(p_csv, index=False)
    pd.DataFrame({"time": times, "temperature": T}).to_csv(t_csv, index=False)

    dvv_data = load_clements_denolle_2023(target, STATION)
    forcings = {
        "precipitation": load_csv_timeseries(p_csv),
        "temperature":   load_csv_timeseries(t_csv),
    }

print(f"Loaded {len(dvv_data)} dv/v samples: "
      f"{dvv_data.index[0]:%Y-%m-%d} \u2192 {dvv_data.index[-1]:%Y-%m-%d}")
print(f"Forcings: {list(forcings.keys())}")

## 3 · Visualise the input data

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 7), sharex=True)

axes[0].plot(dvv_data.index, dvv_data["dvv"] * 1e3, color="steelblue", lw=0.6, alpha=0.8)
axes[0].set_ylabel("dv/v (\u2030)")
axes[0].set_title(f"dv/v \u2014 station {STATION}")

if "temperature" in forcings:
    axes[1].plot(forcings["temperature"].index, forcings["temperature"].values,
                 color="orangered", lw=0.8)
axes[1].set_ylabel("Temperature (\u00b0C)")
axes[1].set_title("Surface temperature")

if "precipitation" in forcings:
    axes[2].bar(forcings["precipitation"].index,
                forcings["precipitation"].values,
                width=1.0, color="royalblue", alpha=0.7)
axes[2].set_ylabel("Precipitation (m/day)")
axes[2].set_title("Daily precipitation")

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_major_locator(mdates.YearLocator(2))

fig.tight_layout()
plt.show()

## 4 · Run the six-phase `codameter` workflow

In [ ]:
result = run_workflow(dvv_data, forcings or None, site)
print(result.summary())

## 5 · Diagnostic six-panel figure

In [ ]:
fig = result.plot_phases()
plt.show()

## 6 · Recovery check (synthetic mode only)

In [ ]:
chi2 = result.phase4.fit.chi2_reduced
print(f"chi\u00b2_red = {chi2:.3f}  (ideal \u2248 1.0)")

if truth is not None:
    print("\nRecovery check:")
    for name, true_val in truth.items():
        try:
            m, s = result.phase4.fit.posterior.marginal(name)
        except KeyError:
            print(f"  {name}: not in posterior (parameter name mismatch?)")
            continue
        z = (m - true_val) / s if s > 0 else float("inf")
        flag = "\u2705 OK" if abs(z) < 4 else "\u274c FAIL"
        print(f"  {name:<12s}  truth={true_val:+.3e}  "
              f"fit={m:+.3e} \u00b1 {s:.2e}  z={z:+.2f}  {flag}")
else:
    print("(Real-data mode \u2014 no truth values to compare against)")

## 7 · Export artefacts to disk

In [ ]:
result.export(OUT_DIR)
print(f"Artefacts written to {OUT_DIR.resolve()}")
for p in sorted(OUT_DIR.iterdir()):
    if p.is_file():
        print(f"  {p.name}")